# 🚗 Centro de Control de Pruebas Vehiculares
### Reporte Técnico de Validación — Frenado (*Braking Performance*) y Consumo de Combustible (*Fuel Economy*)

**Alcance del informe:** Este notebook simula un banco de adquisición de datos (DAQ) conectado al bus CAN de un vehículo de pruebas, integra la información en una base de datos relacional, y expone un panel interactivo para el análisis de:

1. **Desempeño de frenado** (presión hidráulica, desaceleración, activación de ABS, temperatura de disco, distancia de frenado).
2. **Economía de combustible** (flujo instantáneo, consumo promedio en ciclo mixto, hábitos de conducción).

Incluye un motor de reglas de negocio automotriz que emite alertas automáticas cuando los resultados exceden los umbrales de seguridad/eficiencia definidos por ingeniería.

---


## Preparación del entorno
Instalación de dependencias. `sqlite3` es parte de la librería estándar de Python y no requiere instalación.

In [1]:
%pip install -q plotly ipywidgets scipy pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [1]:
import sqlite3
import json
import os
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, Markdown, HTML, clear_output

try:
    from scipy.integrate import cumulative_trapezoid as cumtrapz
except ImportError:
    from scipy.integrate import cumtrapz  # versiones antiguas de scipy

pd.options.mode.chained_assignment = None
np.random.seed(7)

print("Entorno listo. Versiones:")
import plotly, scipy
print(f"  pandas   {pd.__version__}")
print(f"  numpy    {np.__version__}")
print(f"  plotly   {plotly.__version__}")
print(f"  scipy    {scipy.__version__}")
print(f"  ipywidgets {widgets.__version__}")

Entorno listo. Versiones:
  pandas   3.0.5
  numpy    2.3.5
  plotly   6.3.0
  scipy    1.16.3
  ipywidgets 8.1.8


## Simulación de Datos Avanzada (CAN Bus + Sensores)

La función `generate_test_data()` reconstruye 600 segundos de una corrida de pruebas a **10 Hz** (6,000 muestras), combinando:

- **Ciclo de conducción mixto** (ciudad + autopista) construido mediante interpolación de puntos de control de velocidad.
- **5 eventos de frenado explícitos**: 3 frenadas de emergencia (100 → 0 km/h en 3.5 s) y 2 frenadas suaves, con su efecto físico propagado a presión de freno, desaceleración, activación de ABS y temperatura de disco (calentamiento bajo frenada + enfriamiento exponencial hacia la temperatura ambiente).
- **Sistema de motor/combustible** acoplado a RPM, carga del motor y posición del acelerador, con corte de combustible simulado durante desaceleración.

> Nota de ingeniería: la deceleración se deriva numéricamente de la velocidad (`np.gradient`), por lo que todas las señales dependientes (presión de freno, ABS, temperatura de disco) son físicamente consistentes entre sí — igual que en un DAQ real.

In [2]:
def generate_test_data(test_id="TEST_001", seed=42, duration_s=600, freq_hz=10,
                        ambient_temp_c=25.0):
    """
    Simula 600 s de datos de un banco/pista de pruebas a 10 Hz para un vehículo,
    combinando un ciclo de conducción mixto (ciudad + carretera) con 3 frenadas
    de emergencia (100->0 km/h en 3.5 s) y 2 frenadas suaves.

    Devuelve un DataFrame con columnas de chasis, frenos y motor/combustible.
    """
    rng = np.random.default_rng(seed)
    dt = 1.0 / freq_hz
    n = int(duration_s * freq_hz)
    t = np.arange(n) * dt  # segundos

    # ---- 1. Perfil base de velocidad (ciclo mixto ciudad + autopista) ----
    waypoints_t = [0, 15, 40, 55, 75, 95, 110, 130, 150, 200,
                   230, 235,                     # cruce flat -> evento 1 (100->0) en t=235
                   243.5, 270, 320,
                   345, 350,                     # cruce flat -> evento 2 (100->0) en t=350
                   358.5, 400,
                   430, 435,                     # cruce flat -> frenada suave 1 en t=435
                   446, 460, 480, 500,
                   520, 525,                     # cruce flat -> evento 3 (100->0) en t=525
                   533.5, 560,
                   565,                          # cruce flat -> frenada suave 2 en t=565
                   575, 590, 600]
    waypoints_v = [0, 0, 45, 20, 50, 10, 0, 60, 100, 115,
                   100, 100,
                   0, 90, 110,
                   100, 100,
                   0, 95,
                   80, 80,
                   30, 60, 20, 70,
                   100, 100,
                   0, 50,
                   50,
                   10, 20, 15]
    speed_kmh = np.interp(t, waypoints_t, waypoints_v)
    speed_kmh = np.clip(speed_kmh + rng.normal(0, 0.4, n), 0, None)  # ruido de sensor

    # ---- 2. Eventos de frenado explícitos (sobrescriben el perfil base) ----
    # (t_inicio_s, duracion_s, v0_kmh, v1_kmh, tipo)
    brake_events = [
        (235.0, 3.5, 100.0, 0.0, "emergency"),
        (350.0, 3.5, 100.0, 0.0, "emergency"),
        (525.0, 3.5, 100.0, 0.0, "emergency"),
        (435.0, 6.0, 80.0, 30.0, "gentle"),
        (565.0, 5.0, 50.0, 10.0, "gentle"),
    ]
    for (t0, dur, v0, v1, kind) in brake_events:
        mask = (t >= t0) & (t <= t0 + dur)
        if mask.sum() == 0:
            continue
        local_t = t[mask] - t0
        speed_kmh[mask] = v0 + (v1 - v0) * (local_t / dur)
        hold_mask = (t > t0 + dur) & (t <= t0 + dur + 5)
        speed_kmh[hold_mask] = v1 + rng.normal(0, 0.2, hold_mask.sum())

    speed_kmh = np.clip(speed_kmh, 0, None)
    speed_ms = speed_kmh / 3.6

    # ---- 3. Desaceleración (m/s^2), derivada numérica de la velocidad ----
    accel = np.gradient(speed_ms, dt)
    accel = np.clip(accel, -12, 6)

    # ---- 4. Sistema de frenos ----
    braking_mask = accel < -0.4
    brake_pressure = np.zeros(n)
    brake_pressure[braking_mask] = np.clip(-accel[braking_mask] * 20.5, 0, 180)
    brake_pressure += rng.normal(0, 1.0, n) * (brake_pressure > 0)
    brake_pressure = np.clip(brake_pressure, 0, 180)

    abs_active = accel < -6.0  # el ABS interviene en frenadas muy fuertes

    disc_temp = np.empty(n)
    disc_temp[0] = ambient_temp_c
    cooling_rate = 0.003
    heating_gain = 0.55
    for i in range(1, n):
        heat = heating_gain * brake_pressure[i] * dt if brake_pressure[i] > 0 else 0.0
        cool = (disc_temp[i - 1] - ambient_temp_c) * cooling_rate
        disc_temp[i] = disc_temp[i - 1] + heat - cool
    disc_temp = np.clip(disc_temp, ambient_temp_c, 500)

    # ---- 5. Motor / tren motriz ----
    throttle = np.clip(accel * 9.0, 0, 100)
    idle_idx = speed_kmh < 0.5
    throttle[idle_idx] = np.clip(throttle[idle_idx], 0, 8)
    throttle = np.clip(throttle + rng.normal(0, 1.2, n), 0, 100)

    engine_rpm = 800 + speed_kmh * 33 + throttle * 9
    engine_rpm = np.clip(engine_rpm + rng.normal(0, 25, n), 750, 6200)
    engine_rpm[idle_idx] = np.clip(800 + rng.normal(0, 20, idle_idx.sum()), 700, 950)

    engine_load = np.clip(throttle * 0.75 + (engine_rpm / 6200) * 25 + rng.normal(0, 2.5, n), 0, 100)

    coolant_temp = ambient_temp_c + (90 - ambient_temp_c) * (1 - np.exp(-t / 45)) + rng.normal(0, 0.5, n)

    battery_voltage = np.where(engine_rpm > 900, 13.8, 12.6) + rng.normal(0, 0.08, n)
    battery_voltage = np.clip(battery_voltage, 11.8, 14.6)

    # ---- 6. Sistema de combustible ----
    idle_flow = 0.8
    fuel_flow = idle_flow + (engine_rpm / 6200) * 9.5 + (engine_load / 100) * 14.0
    fuel_flow += rng.normal(0, 0.4, n)
    fuel_flow = np.clip(fuel_flow, 0.3, 30)
    fuel_flow[braking_mask] = np.clip(fuel_flow[braking_mask] * 0.4, 0.2, 30)  # corte de combustible

    fuel_start_l = 50.0
    fuel_consumed_cum = np.cumsum(fuel_flow * (dt / 3600.0))
    fuel_level = np.clip(fuel_start_l - fuel_consumed_cum, 0, fuel_start_l)

    df = pd.DataFrame({
        "Test_ID": test_id,
        "Timestamp": t,
        "Vehicle_Speed_kmh": speed_kmh,
        "Engine_RPM": engine_rpm,
        "Coolant_Temp_C": coolant_temp,
        "Battery_Voltage": battery_voltage,
        "Brake_Pressure_Bar": brake_pressure,
        "Brake_Disc_Temp_C": disc_temp,
        "Deceleration_m2s": accel,
        "ABS_Active": abs_active,
        "Throttle_Position_%": throttle,
        "Engine_Load_%": engine_load,
        "Instant_Fuel_Flow_Lh": fuel_flow,
        "Fuel_Level_L": fuel_level,
    })
    return df


# Vista rápida de validación de la función
_preview = generate_test_data()
print(f"Filas generadas: {len(_preview)}  |  Duración: {_preview['Timestamp'].max():.1f} s")
_preview.head()

Filas generadas: 6000  |  Duración: 599.9 s


,Test_ID,Timestamp,Vehicle_Speed_kmh,Engine_RPM,Coolant_Temp_C,Battery_Voltage,Brake_Pressure_Bar,Brake_Disc_Temp_C,Deceleration_m2s,ABS_Active,Throttle_Position_%,Engine_Load_%,Instant_Fuel_Flow_Lh,Fuel_Level_L
0,TEST_001,0.0,0.121887,786.313450,24.140447,12.615064,0.000000,25.000000,-0.338575,False,0.791517,5.835994,2.409512,49.999933
1,TEST_001,0.1,0.000000,832.188193,24.537513,12.487238,0.000000,25.000000,0.247630,False,1.179904,5.649916,2.813615,49.999855
2,TEST_001,0.2,0.300180,835.244029,24.646936,12.598030,0.000000,25.000000,0.522536,False,5.689872,5.907907,3.213918,49.999766
3,TEST_001,0.3,0.376226,797.521953,25.557958,12.536115,7.975356,25.438645,-0.416917,False,0.421951,5.995867,0.975544,49.999739
4,TEST_001,0.4,0.000000,800.908062,25.752480,12.815020,10.931427,26.038557,-0.522536,False,0.000000,8.335532,1.417082,49.999699


### Generación de la Flota de Corridas de Prueba

Se generan **4 corridas** (`Test_ID`) con distintas semillas, condiciones ambientales y objetivos de prueba, simulando una campaña real de validación en pista.

In [3]:
RUN_CONFIG = [
    {"test_id": "TEST_001", "seed": 7,  "ambient_temp_c": 22.0, "tire_pressure_psi": 32, "fuel_type": "E10",    "objective": "Braking"},
    {"test_id": "TEST_002", "seed": 21, "ambient_temp_c": 28.5, "tire_pressure_psi": 33, "fuel_type": "E10",    "objective": "Fuel_Economy"},
    {"test_id": "TEST_003", "seed": 55, "ambient_temp_c": 15.0, "tire_pressure_psi": 30, "fuel_type": "Diesel", "objective": "Braking"},
    {"test_id": "TEST_004", "seed": 99, "ambient_temp_c": 33.0, "tire_pressure_psi": 34, "fuel_type": "Diesel", "objective": "Fuel_Economy"},
]

run_frames = []
conditions_rows = []
for cfg in RUN_CONFIG:
    df_run = generate_test_data(test_id=cfg["test_id"], seed=cfg["seed"], ambient_temp_c=cfg["ambient_temp_c"])
    run_frames.append(df_run)
    conditions_rows.append({
        "Test_ID": cfg["test_id"],
        "Tire_Pressure_PSI": cfg["tire_pressure_psi"],
        "Ambient_Temp_C": cfg["ambient_temp_c"],
        "Fuel_Type": cfg["fuel_type"],
        "Test_Objective": cfg["objective"],
    })

master_df = pd.concat(run_frames, ignore_index=True)
conditions_df = pd.DataFrame(conditions_rows)

print(f"Total de muestras (todas las corridas): {len(master_df):,}")
conditions_df

Total de muestras (todas las corridas): 24,000


,Test_ID,Tire_Pressure_PSI,Ambient_Temp_C,Fuel_Type,Test_Objective
0,TEST_001,32,22.0,E10,Braking
1,TEST_002,33,28.5,E10,Fuel_Economy
2,TEST_003,30,15.0,Diesel,Braking
3,TEST_004,34,33.0,Diesel,Fuel_Economy


## Base de Datos Relacional (SQLite)

Se persisten los datos en dos tablas normalizadas:

- **`test_runs`**: serie de tiempo completa a 10 Hz por `Test_ID` (señales de chasis, freno y motor/combustible).
- **`test_conditions`**: metadatos de cada corrida (`Test_ID` como llave), incluyendo presión de neumáticos, temperatura ambiente, tipo de combustible y objetivo de la prueba — permitiendo hacer `JOIN` entre resultados y condiciones de ensayo.

In [4]:
DB_PATH = "vehicle_test_center.db"

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS test_runs")
cursor.execute("DROP TABLE IF EXISTS test_conditions")

cursor.execute("""
    CREATE TABLE test_runs (
        Test_ID TEXT NOT NULL,
        Timestamp REAL NOT NULL,
        Vehicle_Speed_kmh REAL,
        Engine_RPM REAL,
        Coolant_Temp_C REAL,
        Battery_Voltage REAL,
        Brake_Pressure_Bar REAL,
        Brake_Disc_Temp_C REAL,
        Deceleration_m2s REAL,
        ABS_Active INTEGER,
        "Throttle_Position_%" REAL,
        "Engine_Load_%" REAL,
        Instant_Fuel_Flow_Lh REAL,
        Fuel_Level_L REAL,
        PRIMARY KEY (Test_ID, Timestamp)
    )
""")

cursor.execute("""
    CREATE TABLE test_conditions (
        Test_ID TEXT PRIMARY KEY,
        Tire_Pressure_PSI REAL,
        Ambient_Temp_C REAL,
        Fuel_Type TEXT,
        Test_Objective TEXT,
        FOREIGN KEY (Test_ID) REFERENCES test_runs (Test_ID)
    )
""")
conn.commit()

# Insertar datos (ABS_Active como entero 0/1 para compatibilidad SQL)
_to_insert = master_df.copy()
_to_insert["ABS_Active"] = _to_insert["ABS_Active"].astype(int)
_to_insert.to_sql("test_runs", conn, if_exists="append", index=False)
conditions_df.to_sql("test_conditions", conn, if_exists="append", index=False)
conn.commit()

# Verificación mediante JOIN
check = pd.read_sql("""
    SELECT tc.Test_ID, tc.Test_Objective, tc.Fuel_Type, tc.Ambient_Temp_C,
           COUNT(*) AS n_samples, MAX(tr.Vehicle_Speed_kmh) AS v_max_kmh
    FROM test_conditions tc
    JOIN test_runs tr ON tc.Test_ID = tr.Test_ID
    GROUP BY tc.Test_ID
    ORDER BY tc.Test_ID
""", conn)
print(f"Base de datos '{DB_PATH}' creada con {len(_to_insert):,} registros en test_runs.")
check

Base de datos 'vehicle_test_center.db' creada con 24,000 registros en test_runs.


,Test_ID,Test_Objective,Fuel_Type,Ambient_Temp_C,n_samples,v_max_kmh
0,TEST_001,Braking,E10,22.0,6000,115.871393
1,TEST_002,Fuel_Economy,E10,28.5,6000,115.429340
2,TEST_003,Braking,Diesel,15.0,6000,115.621496
3,TEST_004,Fuel_Economy,Diesel,33.0,6000,115.618920


## Funciones de Análisis y Motor de Reglas de Negocio

Antes de construir el dashboard, se definen las funciones de cálculo de ingeniería y las reglas automotrices que generarán las alertas automáticas.

**Umbrales definidos:**
| Métrica | Umbral | Severidad |
|---|---|---|
| Distancia de frenado 100→0 km/h | > 45 m | 🔴 Seguridad |
| Consumo urbano (velocidad < 60 km/h) | > 12 L/100km | 🟠 Eficiencia |
| Temperatura de disco de freno | > 350 °C | 🟡 Mantenimiento preventivo |

In [5]:
BRAKING_DIST_THRESHOLD_M = 45.0
CITY_CONSUMPTION_THRESHOLD_LP100 = 12.0
DISC_TEMP_THRESHOLD_C = 350.0
CITY_SPEED_LIMIT_KMH = 60.0


def find_emergency_brake_window(df_run, v0_target=100.0, tolerance=10.0):
    """
    Localiza el primer evento de frenado de emergencia (velocidad inicial
    cercana a v0_target) buscando una caída sostenida de velocidad con
    desaceleración fuerte (ABS activo). Devuelve (idx_inicio, idx_fin) o None.
    """
    decel_strong = df_run["Deceleration_m2s"] < -6.0
    if not decel_strong.any():
        return None
    idx_events = df_run.index[decel_strong]
    # agrupar índices contiguos en eventos
    groups, current = [], [idx_events[0]]
    for idx in idx_events[1:]:
        if idx - current[-1] <= 2:
            current.append(idx)
        else:
            groups.append(current)
            current = [idx]
    groups.append(current)

    for g in groups:
        start = max(g[0] - 5, df_run.index.min())
        v_start = df_run.loc[start, "Vehicle_Speed_kmh"]
        if abs(v_start - v0_target) <= tolerance:
            end = g[-1]
            # extender hasta que la velocidad llegue a ~0
            while end + 1 <= df_run.index.max() and df_run.loc[end, "Vehicle_Speed_kmh"] > 0.5:
                end += 1
            return start, end
    return None


def compute_stopping_distance(df_run, v0_target=100.0):
    """Distancia de frenado (m) desde ~100 km/h hasta 0, por integración trapezoidal."""
    window = find_emergency_brake_window(df_run, v0_target=v0_target)
    if window is None:
        return None
    start, end = window
    sub = df_run.loc[start:end]
    speed_ms = sub["Vehicle_Speed_kmh"].to_numpy() / 3.6
    dt = np.diff(sub["Timestamp"].to_numpy(), prepend=sub["Timestamp"].iloc[0] - 0.1).mean()
    distance = cumtrapz(speed_ms, dx=dt, initial=0)
    return float(distance[-1])


def compute_fuel_metrics(df_run):
    """Consumo promedio (L/100km) para todo el ciclo y para el segmento urbano."""
    dt = 0.1
    speed_ms = df_run["Vehicle_Speed_kmh"].to_numpy() / 3.6
    total_dist_km = float(np.sum(speed_ms * dt) / 1000)
    fuel_used_l = float(df_run["Fuel_Level_L"].iloc[0] - df_run["Fuel_Level_L"].iloc[-1])
    avg_lp100km = (fuel_used_l / total_dist_km * 100) if total_dist_km > 0 else np.nan

    city = df_run[df_run["Vehicle_Speed_kmh"] < CITY_SPEED_LIMIT_KMH]
    city_dist_km = float(np.sum((city["Vehicle_Speed_kmh"].to_numpy() / 3.6) * dt) / 1000)
    city_fuel_l = float(np.sum(city["Instant_Fuel_Flow_Lh"].to_numpy() * (dt / 3600)))
    city_lp100km = (city_fuel_l / city_dist_km * 100) if city_dist_km > 0 else np.nan

    return {
        "total_distance_km": total_dist_km,
        "fuel_used_l": fuel_used_l,
        "avg_consumption_lp100km": avg_lp100km,
        "city_consumption_lp100km": city_lp100km,
    }


def evaluate_alerts(df_run):
    """Motor de reglas de negocio automotrices. Devuelve una lista de alertas (dict)."""
    alerts = []

    stopping_distance = compute_stopping_distance(df_run)
    if stopping_distance is not None and stopping_distance > BRAKING_DIST_THRESHOLD_M:
        alerts.append({
            "level": "danger",
            "code": "BRAKE_DISTANCE",
            "message": (f"WARNING: Braking distance ({stopping_distance:.1f} m) exceeds safety "
                        f"threshold ({BRAKING_DIST_THRESHOLD_M:.0f} m). "
                        "Check brake pads and hydraulic circuit."),
        })

    fuel_metrics = compute_fuel_metrics(df_run)
    if not np.isnan(fuel_metrics["city_consumption_lp100km"]) and \
            fuel_metrics["city_consumption_lp100km"] > CITY_CONSUMPTION_THRESHOLD_LP100:
        alerts.append({
            "level": "warning",
            "code": "FUEL_CONSUMPTION",
            "message": (f"WARNING: High fuel consumption detected at low speeds "
                        f"({fuel_metrics['city_consumption_lp100km']:.1f} L/100km). "
                        "Review injection timing or O2 sensors."),
        })

    hot_mask = df_run["Brake_Disc_Temp_C"] > DISC_TEMP_THRESHOLD_C
    if hot_mask.any():
        hot_rows = df_run.loc[hot_mask, ["Timestamp", "Brake_Disc_Temp_C"]]
        peak_row = hot_rows.loc[hot_rows["Brake_Disc_Temp_C"].idxmax()]
        alerts.append({
            "level": "caution",
            "code": "BRAKE_FADE_RISK",
            "message": (f"MAINTENANCE: Brake disc temperature reached "
                        f"{peak_row['Brake_Disc_Temp_C']:.0f}\u00b0C at t={peak_row['Timestamp']:.1f}s "
                        f"(> {DISC_TEMP_THRESHOLD_C:.0f}\u00b0C). Flag for brake material inspection "
                        "(potential brake fade)."),
        })

    return alerts, stopping_distance, fuel_metrics


_ALERT_STYLE = {
    "danger":  ("#FDECEA", "#B71C1C", "\U0001F534"),
    "warning": ("#FFF4E5", "#8A4B00", "\U0001F7E0"),
    "caution": ("#FFFBEA", "#7A5B00", "\U0001F7E1"),
}


def render_alerts_html(alerts):
    if not alerts:
        return "<div style='padding:10px 14px;background:#EAF7EE;border-left:5px solid #2E7D32;" \
               "border-radius:4px;font-family:sans-serif;color:#1B5E20;'>" \
               "\u2705 All parameters within safety and efficiency thresholds.</div>"
    blocks = []
    for a in alerts:
        bg, fg, icon = _ALERT_STYLE.get(a["level"], ("#eee", "#333", "\u26A0\uFE0F"))
        blocks.append(
            f"<div style='padding:10px 14px;margin-bottom:6px;background:{bg};"
            f"border-left:5px solid {fg};border-radius:4px;font-family:sans-serif;"
            f"color:{fg};'>{icon} <b>{a['code']}</b> — {a['message']}</div>"
        )
    return "".join(blocks)


print("Funciones de análisis y motor de reglas de negocio cargadas correctamente.")

Funciones de análisis y motor de reglas de negocio cargadas correctamente.


## Dashboard Interactivo de Control de Pruebas

Panel con **3 pestañas** construido con `ipywidgets` + `Plotly`. Todas las gráficas son completamente interactivas.

Usa los selectores superiores para elegir el **objetivo de prueba** y el **`Test_ID`**; las tres pestañas se actualizan de forma simultánea.

In [10]:
# ---------------------------------------------------------------
# Controles del dashboard
# ---------------------------------------------------------------
objective_dropdown = widgets.Dropdown(
    options=["All", "Braking", "Fuel_Economy"],
    value="All",
    description="Objetivo:",
    style={"description_width": "initial"},
)

test_id_dropdown = widgets.Dropdown(
    options=sorted(master_df["Test_ID"].unique().tolist()),
    value=sorted(master_df["Test_ID"].unique().tolist())[0],
    description="Test_ID:",
    style={"description_width": "initial"},
)

header_label = widgets.HTML("<h3 style='margin-bottom:0'>3.0 Panel de Control</h3>")

out_overview = widgets.Output()
out_braking = widgets.Output()
out_fuel = widgets.Output()


def get_run_data(test_id):
    return master_df[master_df["Test_ID"] == test_id].reset_index(drop=True)


def get_conditions(test_id):
    row = conditions_df[conditions_df["Test_ID"] == test_id]
    return row.iloc[0].to_dict() if len(row) else {}


def big_number_html(label, value_str, sub="", color="#0B3D91"):
    return f"""
    <div style='font-family:sans-serif;text-align:center;padding:14px;
                border:1px solid #e0e0e0;border-radius:10px;background:#FAFBFF;'>
      <div style='font-size:13px;color:#666;letter-spacing:.04em;text-transform:uppercase;'>{label}</div>
      <div style='font-size:40px;font-weight:700;color:{color};line-height:1.1;margin-top:4px;'>{value_str}</div>
      <div style='font-size:12px;color:#888;margin-top:2px;'>{sub}</div>
    </div>
    """


def kpi_gauge_figure(df_run):
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{"type": "indicator"}, {"type": "indicator"}]],
        subplot_titles=("Battery Voltage (V)", "Coolant Temp (\u00b0C)"),
    )
    batt = df_run["Battery_Voltage"].iloc[-1]
    coolant = df_run["Coolant_Temp_C"].iloc[-1]
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=batt,
        number={"suffix": " V", "font": {"size": 30}},
        gauge={
            "axis": {"range": [11.5, 15]},
            "bar": {"color": "#0B3D91"},
            "steps": [
                {"range": [11.5, 12.2], "color": "#FDECEA"},
                {"range": [12.2, 14.6], "color": "#EAF7EE"},
                {"range": [14.6, 15], "color": "#FFF4E5"},
            ],
        },
    ), row=1, col=1)
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=coolant,
        number={"suffix": " \u00b0C", "font": {"size": 30}},
        gauge={
            "axis": {"range": [0, 130]},
            "bar": {"color": "#B71C1C"},
            "steps": [
                {"range": [0, 80], "color": "#EAF7EE"},
                {"range": [80, 105], "color": "#FFF4E5"},
                {"range": [105, 130], "color": "#FDECEA"},
            ],
        },
    ), row=1, col=2)
    fig.update_layout(height=260, margin=dict(l=30, r=30, t=50, b=10))
    return fig


def speed_rpm_figure(df_run):
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(
        x=df_run["Timestamp"], y=df_run["Vehicle_Speed_kmh"],
        name="Vehicle Speed (km/h)", line=dict(color="#0B3D91", width=1.6),
    ), secondary_y=False)
    fig.add_trace(go.Scatter(
        x=df_run["Timestamp"], y=df_run["Engine_RPM"],
        name="Engine RPM", line=dict(color="#E8730A", width=1.2),
        opacity=0.75,
    ), secondary_y=True)
    fig.update_xaxes(title_text="Tiempo (s)")
    fig.update_yaxes(title_text="Velocidad (km/h)", secondary_y=False)
    fig.update_yaxes(title_text="RPM", secondary_y=True)
    fig.update_layout(
        title="3.1 Velocidad del Vehículo vs. RPM del Motor",
        height=420, hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(t=60, b=40),
    )
    return fig


def decel_pressure_figure(df_run):
    colors = np.where(df_run["ABS_Active"], "#D32F2F", "#1976D2")
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=df_run["Brake_Pressure_Bar"], y=df_run["Deceleration_m2s"],
        mode="markers",
        marker=dict(color=colors, size=4, opacity=0.6),
        text=np.where(df_run["ABS_Active"], "ABS ACTIVE", "ABS inactive"),
        hovertemplate="Presión: %{x:.1f} bar<br>Deceleración: %{y:.2f} m/s²<br>%{text}<extra></extra>",
    ))
    fig.update_layout(
        title="3.2 Análisis de Desaceleración vs. Presión de Freno (rojo = ABS activo)",
        xaxis_title="Brake Pressure (bar)",
        yaxis_title="Deceleration (m/s²)",
        height=380, margin=dict(t=60, b=40),
    )
    return fig


def disc_temp_figure(df_run):
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_run["Timestamp"], y=df_run["Brake_Disc_Temp_C"],
        name="Brake Disc Temp (\u00b0C)", line=dict(color="#B71C1C", width=1.4),
        fill="tozeroy", fillcolor="rgba(183,28,28,0.08)",
    ))
    fig.add_hline(y=DISC_TEMP_THRESHOLD_C, line_dash="dash", line_color="#8A4B00",
                   annotation_text=f"Umbral de Brake Fade ({DISC_TEMP_THRESHOLD_C:.0f}\u00b0C)",
                   annotation_position="top left")
    fig.update_layout(
        title="3.3 Temperatura de Disco de Freno (detección de Brake Fade)",
        xaxis_title="Tiempo (s)", yaxis_title="Temperatura (\u00b0C)",
        height=380, margin=dict(t=60, b=40),
    )
    return fig


def fuel_flow_speed_figure(df_run):
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(
        x=df_run["Timestamp"], y=df_run["Vehicle_Speed_kmh"],
        name="Vehicle Speed (km/h)", line=dict(color="#0B3D91", width=1.4),
    ), secondary_y=False)
    fig.add_trace(go.Scatter(
        x=df_run["Timestamp"], y=df_run["Instant_Fuel_Flow_Lh"],
        name="Instant Fuel Flow (L/h)", line=dict(color="#2E7D32", width=1.4),
        opacity=0.85,
    ), secondary_y=True)
    fig.update_xaxes(title_text="Tiempo (s)")
    fig.update_yaxes(title_text="Velocidad (km/h)", secondary_y=False)
    fig.update_yaxes(title_text="Flujo de Combustible (L/h)", secondary_y=True)
    fig.update_layout(
        title="3.4 Flujo de Combustible Instantáneo vs. Velocidad",
        height=400, hovermode="x unified",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        margin=dict(t=60, b=40),
    )
    return fig


def throttle_load_figure(df_run):
    aggressive = (df_run["Throttle_Position_%"] > 55) & (df_run["Engine_Load_%"] > 55)
    colors = np.where(aggressive, "#D32F2F", "#2E7D32")
    fig = go.Figure()
    fig.add_trace(go.Scattergl(
        x=df_run["Throttle_Position_%"], y=df_run["Engine_Load_%"],
        mode="markers",
        marker=dict(color=colors, size=4, opacity=0.55),
        text=np.where(aggressive, "Conducción agresiva", "Conducción eficiente"),
        hovertemplate="Throttle: %{x:.0f}%%<br>Load: %{y:.0f}%%<br>%{text}<extra></extra>",
    ))
    fig.update_layout(
        title="3.5 Posición del Acelerador vs. Carga del Motor (rojo = conducción agresiva)",
        xaxis_title="Throttle Position (%)", yaxis_title="Engine Load (%)",
        height=380, margin=dict(t=60, b=40),
    )
    return fig


print("Funciones de graficación definidas.")

Funciones de graficación definidas.


In [11]:
# ---------------------------------------------------------------
#  Lógica de actualización del dashboard
# ---------------------------------------------------------------
def refresh_test_id_options(*_):
    obj = objective_dropdown.value
    if obj == "All":
        ids = sorted(master_df["Test_ID"].unique().tolist())
    else:
        ids = sorted(conditions_df.loc[conditions_df["Test_Objective"] == obj, "Test_ID"].tolist())
    if not ids:
        ids = sorted(master_df["Test_ID"].unique().tolist())
    current = test_id_dropdown.value
    test_id_dropdown.options = ids
    test_id_dropdown.value = current if current in ids else ids[0]


def render_overview(df_run, cond):
    with out_overview:
        clear_output(wait=True)
        info = (f"<div style='font-family:sans-serif;color:#444;margin-bottom:8px;'>"
                f"<b>Test_ID:</b> {df_run['Test_ID'].iloc[0]} &nbsp;|&nbsp; "
                f"<b>Objetivo:</b> {cond.get('Test_Objective','-')} &nbsp;|&nbsp; "
                f"<b>Combustible:</b> {cond.get('Fuel_Type','-')} &nbsp;|&nbsp; "
                f"<b>Presión de Neumáticos:</b> {cond.get('Tire_Pressure_PSI','-')} PSI &nbsp;|&nbsp; "
                f"<b>Temp. Ambiente:</b> {cond.get('Ambient_Temp_C','-')}\u00b0C</div>")
        display(HTML(info))
        display(go.FigureWidget(speed_rpm_figure(df_run)))
        display(go.FigureWidget(kpi_gauge_figure(df_run)))


def render_braking(df_run):
    with out_braking:
        clear_output(wait=True)
        alerts, stopping_distance, _ = evaluate_alerts(df_run)
        brake_alerts = [a for a in alerts if a["code"] in ("BRAKE_DISTANCE", "BRAKE_FADE_RISK")]

        dist_str = f"{stopping_distance:.1f} m" if stopping_distance is not None else "N/D"
        dist_color = "#B71C1C" if (stopping_distance and stopping_distance > BRAKING_DIST_THRESHOLD_M) else "#2E7D32"
        display(HTML(big_number_html(
            "Distancia de Frenado 100 → 0 km/h",
            dist_str,
            sub=f"Umbral de seguridad: {BRAKING_DIST_THRESHOLD_M:.0f} m  (integración trapezoidal)",
            color=dist_color,
        )))
        display(HTML("<div style='height:10px'></div>"))
        display(HTML(render_alerts_html(brake_alerts)))
        display(go.FigureWidget(decel_pressure_figure(df_run)))
        display(go.FigureWidget(disc_temp_figure(df_run)))


def render_fuel(df_run):
    with out_fuel:
        clear_output(wait=True)
        alerts, _, fuel_metrics = evaluate_alerts(df_run)
        fuel_alerts = [a for a in alerts if a["code"] == "FUEL_CONSUMPTION"]

        avg_c = fuel_metrics["avg_consumption_lp100km"]
        city_c = fuel_metrics["city_consumption_lp100km"]
        color = "#B71C1C" if (not np.isnan(city_c) and city_c > CITY_CONSUMPTION_THRESHOLD_LP100) else "#2E7D32"

        kpi_row = widgets.HBox([
            widgets.HTML(big_number_html("Consumo Promedio (ciclo completo)",
                                          f"{avg_c:.1f} L/100km",
                                          sub=f"Distancia total: {fuel_metrics['total_distance_km']:.1f} km")),
            widgets.HTML(big_number_html("Consumo en Ciudad (< 60 km/h)",
                                          f"{city_c:.1f} L/100km",
                                          sub=f"Umbral: {CITY_CONSUMPTION_THRESHOLD_LP100:.0f} L/100km",
                                          color=color)),
        ])
        display(kpi_row)
        display(HTML("<div style='height:10px'></div>"))
        display(HTML(render_alerts_html(fuel_alerts)))
        display(go.FigureWidget(fuel_flow_speed_figure(df_run)))
        display(go.FigureWidget(throttle_load_figure(df_run)))


def on_change(change):
    test_id = test_id_dropdown.value
    df_run = get_run_data(test_id)
    cond = get_conditions(test_id)
    render_overview(df_run, cond)
    render_braking(df_run)
    render_fuel(df_run)


objective_dropdown.observe(refresh_test_id_options, names="value")
test_id_dropdown.observe(on_change, names="value")
objective_dropdown.observe(on_change, names="value")

# Render inicial
on_change(None)

tabs = widgets.Tab(children=[out_overview, out_braking, out_fuel])
tabs.set_title(0, "📊 Visión General")
tabs.set_title(1, "🛑 Frenado (Safety First)")
tabs.set_title(2, "⛽ Consumo (Eficiencia)")

controls = widgets.HBox([objective_dropdown, test_id_dropdown])
display(widgets.VBox([header_label, controls, tabs]))

## Resumen Ejecutivo y Alertas Automáticas

A continuación se evalúa el motor de reglas de negocio sobre **cada corrida** de la base de datos y se consolida un resumen ejecutivo. Esto simula el reporte que un ingeniero de validación enviaría al equipo de Producto/Calidad al cierre de una campaña de pruebas.

In [13]:
# ---------------------------------------------------------------
# Evaluación de reglas de negocio para toda la flota de pruebas
# ---------------------------------------------------------------
executive_records = []

for test_id in sorted(master_df["Test_ID"].unique()):
    df_run = get_run_data(test_id)
    cond = get_conditions(test_id)
    alerts, stopping_distance, fuel_metrics = evaluate_alerts(df_run)

    display(Markdown(f"### {test_id.split('_')[-1]} Test_ID: `{test_id}`  "
                      f"({cond.get('Test_Objective','-')} — {cond.get('Fuel_Type','-')})"))
    display(HTML(render_alerts_html(alerts)))

    executive_records.append({
        "test_id": test_id,
        "test_objective": cond.get("Test_Objective"),
        "fuel_type": cond.get("Fuel_Type"),
        "ambient_temp_c": cond.get("Ambient_Temp_C"),
        "tire_pressure_psi": cond.get("Tire_Pressure_PSI"),
        "max_decel_ms2": float(df_run["Deceleration_m2s"].min()),
        "stopping_distance_100_0_m": stopping_distance,
        "avg_fuel_consumption_lp100km": fuel_metrics["avg_consumption_lp100km"],
        "city_fuel_consumption_lp100km": fuel_metrics["city_consumption_lp100km"],
        "max_brake_temp_c": float(df_run["Brake_Disc_Temp_C"].max()),
        "abs_activations": int(df_run["ABS_Active"].sum()),
        "n_alerts": len(alerts),
        "alerts": [a["message"] for a in alerts],
    })

executive_df = pd.DataFrame(executive_records)
executive_df

### 001 Test_ID: `TEST_001`  (Braking — E10)

### 002 Test_ID: `TEST_002`  (Fuel_Economy — E10)

### 003 Test_ID: `TEST_003`  (Braking — Diesel)

### 004 Test_ID: `TEST_004`  (Fuel_Economy — Diesel)

,test_id,test_objective,fuel_type,ambient_temp_c,tire_pressure_psi,max_decel_ms2,stopping_distance_100_0_m,avg_fuel_consumption_lp100km,city_fuel_consumption_lp100km,max_brake_temp_c,abs_activations,n_alerts,alerts
0,TEST_001,Braking,E10,22.0,32,-7.936508,59.721016,10.165014,15.706162,417.971868,102,3,[WARNING: Braking distance (59.7 m) exceeds sa...
1,TEST_002,Fuel_Economy,E10,28.5,33,-7.936508,59.749433,10.183056,15.718382,418.301658,102,3,[WARNING: Braking distance (59.7 m) exceeds sa...
2,TEST_003,Braking,Diesel,15.0,30,-7.936508,59.738171,10.137014,15.692768,419.719122,102,3,[WARNING: Braking distance (59.7 m) exceeds sa...
3,TEST_004,Fuel_Economy,Diesel,33.0,34,-7.936508,59.705582,10.145903,15.685485,436.874452,102,3,[WARNING: Braking distance (59.7 m) exceeds sa...


### Tabla Consolidada de Indicadores Clave

In [14]:
display_cols = ["test_id", "test_objective", "max_decel_ms2", "stopping_distance_100_0_m",
                "avg_fuel_consumption_lp100km", "city_fuel_consumption_lp100km",
                "max_brake_temp_c", "abs_activations", "n_alerts"]
executive_df[display_cols].style.format({
    "max_decel_ms2": "{:.2f}",
    "stopping_distance_100_0_m": "{:.1f}",
    "avg_fuel_consumption_lp100km": "{:.1f}",
    "city_fuel_consumption_lp100km": "{:.1f}",
    "max_brake_temp_c": "{:.0f}",
}, na_rep="N/D").background_gradient(subset=["stopping_distance_100_0_m"], cmap="Reds") \
  .background_gradient(subset=["max_brake_temp_c"], cmap="Oranges")

,test_id,test_objective,max_decel_ms2,stopping_distance_100_0_m,avg_fuel_consumption_lp100km,city_fuel_consumption_lp100km,max_brake_temp_c,abs_activations,n_alerts
0,TEST_001,Braking,-7.94,59.7,10.2,15.7,418,102,3
1,TEST_002,Fuel_Economy,-7.94,59.7,10.2,15.7,418,102,3
2,TEST_003,Braking,-7.94,59.7,10.1,15.7,420,102,3
3,TEST_004,Fuel_Economy,-7.94,59.7,10.1,15.7,437,102,3


## Exportación del Resumen Ejecutivo (JSON)

Se guarda un resumen ejecutivo en `/reports/executive_summary.json` con las métricas clave de la **corrida principal** (`TEST_001`, la más completa: incluye los 3 frenados de emergencia y los 2 frenados suaves del ciclo mixto), además del detalle por corrida de toda la flota de pruebas.

In [15]:
# ---------------------------------------------------------------
# Construcción y exportación del JSON ejecutivo
# ---------------------------------------------------------------
os.makedirs("reports", exist_ok=True)

primary_test_id = "TEST_001"
primary_record = next(r for r in executive_records if r["test_id"] == primary_test_id)

executive_summary = {
    "report_generated_at": datetime.now().isoformat(timespec="seconds"),
    "primary_test_id": primary_test_id,
    "safety_thresholds": {
        "braking_distance_m": BRAKING_DIST_THRESHOLD_M,
        "city_fuel_consumption_lp100km": CITY_CONSUMPTION_THRESHOLD_LP100,
        "brake_disc_temp_c": DISC_TEMP_THRESHOLD_C,
    },
    "primary_test_metrics": {
        "max_decel": primary_record["max_decel_ms2"],
        "stopping_distance_100_0": primary_record["stopping_distance_100_0_m"],
        "avg_fuel_consumption": primary_record["avg_fuel_consumption_lp100km"],
        "max_brake_temp": primary_record["max_brake_temp_c"],
    },
    "fleet_summary": executive_records,
}

report_path = os.path.join("reports", "executive_summary.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(executive_summary, f, indent=2, ensure_ascii=False)

print(f"Resumen ejecutivo guardado en: {os.path.abspath(report_path)}\n")
print(json.dumps(executive_summary["primary_test_metrics"], indent=2))

Resumen ejecutivo guardado en: C:\Users\kike2\Documents\GitHub\Control de Pruebas Vehiculares\reports\executive_summary.json

{
  "max_decel": -7.936507936509756,
  "stopping_distance_100_0": 59.721015869614526,
  "avg_fuel_consumption": 10.165014329557218,
  "max_brake_temp": 417.9718684523185
}


---
### Conclusiones del Reporte

- El motor de reglas evalúa automáticamente **cada corrida** contra los umbrales de seguridad, eficiencia y mantenimiento definidos en la Sección 3, sin intervención manual.
- La distancia de frenado se calcula por **integración trapezoidal** (`scipy.integrate.cumulative_trapezoid`, con fallback a `cumtrapz`/`cumsum` según la versión de SciPy disponible), garantizando consistencia física con la señal de velocidad medida.
- El archivo `/reports/executive_summary.json` puede integrarse directamente a un pipeline de CI/CD de validación (ej. GitHub Actions) para bloquear la liberación de un vehículo/calibración si algún `n_alerts > 0` en pruebas de seguridad críticas.